In [13]:
# ------------------- CREATE eICU ANEMIA VALIDATION DATASET WITH UNIFIED DRUG SPACE -------------------
import pandas as pd
import numpy as np
import re
import os
from tqdm import tqdm
from rapidfuzz import process

# ------------------- Alternative: Use String Matching with MIMIC Vocabulary -------------------
# Instead of RxNorm API (which may be unavailable), use fuzzy matching to MIMIC drugs
# This creates a unified drug space without external API calls

print("="*80)
print("STEP 1: Load MIMIC Dataset and Create Drug Vocabulary")
print("="*80)

mimic_anemia_path = r'......user_drug_rating_visit_anemia.csv'
mimic_anemia_df = pd.read_csv(mimic_anemia_path)

# Create MIMIC drug vocabulary (standardized names)
mimic_unique_drugs = mimic_anemia_df['item'].str.lower().unique()
print(f"MIMIC unique drugs: {len(mimic_unique_drugs)}")

# Create a mapping from cleaned drug name to canonical MIMIC name
def canonicalize_drug_name(name):
    """Create canonical drug name by removing common suffixes and special characters"""
    name = str(name).lower().strip()
    # Remove common suffixes
    name = re.sub(r'\s*(?:tablet|capsule|injection|solution|suspension|cream|ointment|patch|syrup|drop|oral|iv|inhalation|nebulizer|flush|lotion|gel|spray|powder|suppository)\b', '', name)
    # Remove dosage
    name = re.sub(r'\b\d+(?:\.\d+)?\s*(?:mg|ml|mcg|g|mg/ml|mcg/ml|%|meq|unit|units)\b', '', name)
    # Remove special characters
    name = re.sub(r'[^\w\s]', ' ', name)
    # Remove extra spaces
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Create canonical MIMIC drug mapping
mimic_canonical_map = {}
for drug in mimic_unique_drugs:
    canonical = canonicalize_drug_name(drug)
    if canonical not in mimic_canonical_map:
        mimic_canonical_map[canonical] = drug
    else:
        # Keep the shorter name
        if len(drug) < len(mimic_canonical_map[canonical]):
            mimic_canonical_map[canonical] = drug

print(f"Canonical MIMIC drugs: {len(mimic_canonical_map)}")

# ------------------- Load eICU Data -------------------
print("\n" + "="*80)
print("STEP 2: Load eICU Data")
print("="*80)

eicu_path = r'..........eicu-collaborative-research-database-2.0'

# Load diagnosis data
diagnosis_eicu = pd.read_csv(os.path.join(eicu_path, 'diagnosis.csv.gz'), low_memory=False)
print(f"diagnosis.csv loaded: {diagnosis_eicu.shape}")

# Load medication data
medication_eicu = pd.read_csv(os.path.join(eicu_path, 'medication.csv.gz'), low_memory=False)
print(f"medication.csv loaded: {medication_eicu.shape}")

# Load admission drugs
admission_drugs = pd.read_csv(os.path.join(eicu_path, 'admissionDrug.csv.gz'), low_memory=False)
print(f"admissionDrug.csv loaded: {admission_drugs.shape}")

# -------------------- Clean ICD9 codes --------------------
def extract_icd9_code(icd9_value):
    if pd.isna(icd9_value):
        return None
    icd9_str = str(icd9_value).split(',')[0].split()[0].strip()
    icd9_str = icd9_str.split('.')[0]
    icd9_str = re.sub(r'[^0-9]', '', icd9_str)
    return icd9_str if len(icd9_str) >= 3 else None

diagnosis_eicu['ICD9_CODE'] = diagnosis_eicu['icd9code'].apply(extract_icd9_code)
diagnosis_eicu = diagnosis_eicu.dropna(subset=['ICD9_CODE'])

# -------------------- Get ANEMIA patients from eICU --------------------
anemia_icd9_prefixes = ['280', '281', '282', '283', '284', '285']

diagnoses_anemia_eicu = diagnosis_eicu[
    diagnosis_eicu['ICD9_CODE'].str[:3].isin(anemia_icd9_prefixes)
]
patients_anemia_eicu = set(diagnoses_anemia_eicu['patientunitstayid'].unique())
print(f"Anemia patients in eICU: {len(patients_anemia_eicu)}")

# -------------------- Clean drug names function --------------------
def clean_drug_name_eicu(name):
    if pd.isnull(name):
        return ""
    name = str(name).lower().strip()
    # Remove dosage information
    name = re.sub(r'\b\d+(?:\.\d+)?\s*(?:mg|ml|mcg|g|mcg|tablet|tab|capsule|cap|inj|solution|suspension|flush|spray)\b', '', name)
    name = re.sub(r'\b\d+\s*(?:%|mg/ml|mcg/ml)\b', '', name)
    # Remove common suffixes
    name = re.sub(r'\s*(?:tab|caps|inj|soln|sol|susp|flush|spray)\s*', ' ', name)
    # Remove punctuation
    name = re.sub(r'[^\w\s]', ' ', name)
    # Remove extra spaces
    name = re.sub(r'\s+', ' ', name)
    # Remove common dosing schedules
    name = re.sub(r'\b(?:q\d+[h]?|bid|tid|qid|qd|prn|daily|hour|weekly|monthly|sliding scale)\b', '', name)
    return name.strip()

def canonicalize_drug_name_eicu(name):
    """Create canonical drug name for eICU (matching MIMIC format)"""
    name = clean_drug_name_eicu(name)
    # Remove brand names and keep generic-like form
    name = re.sub(r'\b(?:nf|ec|sr|er|xr|cr|dr|hr)\b', '', name)
    # Remove strength indicators
    name = re.sub(r'\b(?:forte|double|extra|super|plus)\b', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# -------------------- Process eICU medications --------------------
def get_patient_medications(patient_ids):
    medications = []
    
    # From medication.csv
    med_filtered = medication_eicu[medication_eicu['patientunitstayid'].isin(patient_ids)]
    for _, row in med_filtered.iterrows():
        drug_name = row.get('drugname', '')
        if pd.notna(drug_name) and str(drug_name).strip():
            cleaned = clean_drug_name_eicu(drug_name)
            if cleaned and len(cleaned) > 2:
                medications.append({
                    'patientunitstayid': row['patientunitstayid'],
                    'drug_name': cleaned
                })
    
    # From admissionDrug.csv
    adm_filtered = admission_drugs[admission_drugs['patientunitstayid'].isin(patient_ids)]
    for _, row in adm_filtered.iterrows():
        drug_name = row.get('drugname', row.get('medicationname', row.get('drug', '')))
        if pd.notna(drug_name) and str(drug_name).strip():
            cleaned = clean_drug_name_eicu(drug_name)
            if cleaned and len(cleaned) > 2:
                medications.append({
                    'patientunitstayid': row['patientunitstayid'],
                    'drug_name': cleaned
                })
    
    return pd.DataFrame(medications)

# Get medications for anemia patients
print("\nExtracting medications for anemia patients...")
medications_df = get_patient_medications(patients_anemia_eicu)
print(f"Total medication records: {len(medications_df):,}")

# Remove duplicates
medications_df = medications_df.drop_duplicates(subset=['patientunitstayid', 'drug_name'])
print(f"After removing duplicates: {len(medications_df):,}")

# Filter out empty drug names
medications_df = medications_df[medications_df['drug_name'].str.len() > 2]
print(f"After filtering empty names: {len(medications_df):,}")

# -------------------- Fuzzy Match eICU Drugs to MIMIC Vocabulary --------------------
print("\n" + "="*80)
print("STEP 3: Match eICU Drugs to MIMIC Vocabulary")
print("="*80)

# Get unique eICU drugs
eicu_unique_drugs = medications_df['drug_name'].unique()
print(f"Unique eICU drugs before matching: {len(eicu_unique_drugs)}")

# Create list of MIMIC canonical names for matching
mimic_canonical_list = list(mimic_canonical_map.keys())

# Perform fuzzy matching
matched_drugs = {}
match_count = 0
for drug in tqdm(eicu_unique_drugs, desc="Matching eICU drugs to MIMIC"):
    # First try exact match on canonical form
    canonical_eicu = canonicalize_drug_name_eicu(drug)
    
    if canonical_eicu in mimic_canonical_map:
        matched_drugs[drug] = mimic_canonical_map[canonical_eicu]
        match_count += 1
    else:
        # Try fuzzy matching
        match = process.extractOne(canonical_eicu, mimic_canonical_list, score_cutoff=80)
        if match:
            matched_drugs[drug] = mimic_canonical_map[match[0]]
            match_count += 1
        else:
            # Try with original drug name
            match = process.extractOne(drug, mimic_unique_drugs, score_cutoff=80)
            if match:
                matched_drugs[drug] = match[0]
                match_count += 1

print(f"Matched {match_count}/{len(eicu_unique_drugs)} ({100*match_count/len(eicu_unique_drugs):.1f}%)")

# Add matched MIMIC drug to dataframe
medications_df['mimic_drug'] = medications_df['drug_name'].map(matched_drugs)

# Keep only drugs that matched
medications_matched = medications_df[medications_df['mimic_drug'].notna()].copy()
print(f"After filtering to matched drugs: {len(medications_matched):,} interactions")

# -------------------- Create User-Drug Interaction Matrix --------------------
user_drug_eicu = medications_matched.groupby(['patientunitstayid', 'mimic_drug']).size().reset_index(name='rating')
user_drug_eicu['rating'] = 1

# Rename columns
user_drug_eicu = user_drug_eicu.rename(columns={
    'patientunitstayid': 'user',
    'mimic_drug': 'item'
})

# -------------------- Filter low-frequency items --------------------
MIN_DRUG_FREQUENCY = 5
drug_counts = user_drug_eicu['item'].value_counts()
frequent_drugs = drug_counts[drug_counts >= MIN_DRUG_FREQUENCY].index
user_drug_eicu = user_drug_eicu[user_drug_eicu['item'].isin(frequent_drugs)]
print(f"After filtering drugs with <{MIN_DRUG_FREQUENCY} occurrences: {user_drug_eicu['item'].nunique()} unique drugs")

# Filter patients with at least 2 drugs
patient_counts = user_drug_eicu['user'].value_counts()
active_patients = patient_counts[patient_counts >= 2].index
user_drug_eicu = user_drug_eicu[user_drug_eicu['user'].isin(active_patients)]
print(f"After filtering patients with <2 drugs: {user_drug_eicu['user'].nunique()} patients")

# -------------------- Print Statistics --------------------
print("\n" + "="*70)
print("eICU ANEMIA DATASET STATISTICS (MIMIC-Matched)")
print("="*70)
print(f"Total anemia patients: {user_drug_eicu['user'].nunique():,}")
print(f"Total interactions: {len(user_drug_eicu):,}")
print(f"Unique drugs (MIMIC-matched): {user_drug_eicu['item'].nunique():,}")
print(f"Average drugs per patient: {len(user_drug_eicu) / user_drug_eicu['user'].nunique():.2f}")

print("\nFirst 10 rows:")
print(user_drug_eicu.head(10))

# -------------------- Save to CSV --------------------
output_path = r'............user_drug_rating_visit_eicu_anemia.csv'
user_drug_eicu.to_csv(output_path, index=False)
print(f"\n✓ Saved: {output_path}")

# -------------------- Verify Drug Overlap --------------------
print("\n" + "="*70)
print("DRUG OVERLAP VERIFICATION")
print("="*70)

mimic_drugs = set(mimic_anemia_df['item'].str.lower().unique())
eicu_drugs_matched = set(user_drug_eicu['item'].str.lower().unique())
overlap = mimic_drugs.intersection(eicu_drugs_matched)

print(f"MIMIC unique drugs: {len(mimic_drugs):,}")
print(f"eICU unique drugs (matched): {len(eicu_drugs_matched):,}")
print(f"Common drugs: {len(overlap):,}")
print(f"Overlap % of MIMIC: {100*len(overlap)/len(mimic_drugs):.1f}%")
print(f"Overlap % of eICU: {100*len(overlap)/len(eicu_drugs_matched):.1f}%")

# Top 20 common drugs
print("\nTop 20 Common Drugs between MIMIC-III and eICU:")
common_drugs_list = list(overlap)[:20]
for i, drug in enumerate(sorted(common_drugs_list), 1):
    print(f"  {i:2d}. {drug}")

# Unmatched drugs summary
matched_count = len(eicu_drugs_matched)
unmatched_count = len(eicu_unique_drugs) - matched_count
print(f"\nUnmatched eICU drugs: {unmatched_count}/{len(eicu_unique_drugs)} ({100*unmatched_count/len(eicu_unique_drugs):.1f}%)")

# -------------------- Quality Checks --------------------
print("\n" + "="*70)
print("QUALITY CHECKS")
print("="*70)

# Check for missing values
print(f"\nMissing values:")
print(f"  Users: {user_drug_eicu['user'].isna().sum()}")
print(f"  Items: {user_drug_eicu['item'].isna().sum()}")

# Distribution of interactions per patient
interactions_per_patient = user_drug_eicu.groupby('user').size()
print(f"\nInteractions per patient:")
print(f"  Min: {interactions_per_patient.min()}")
print(f"  Max: {interactions_per_patient.max()}")
print(f"  Mean: {interactions_per_patient.mean():.1f}")
print(f"  Median: {interactions_per_patient.median():.1f}")

# Top 20 drugs in eICU anemia dataset
top_drugs_eicu = user_drug_eicu['item'].value_counts().head(20)
print(f"\nTop 20 drugs in eICU anemia dataset (MIMIC-matched):")
for i, (drug, count) in enumerate(top_drugs_eicu.items(), 1):
    print(f"  {i:2d}. {drug}: {count:,} patients")

print("\n" + "="*70)
print("✅ eICU ANEMIA VALIDATION DATASET CREATED SUCCESSFULLY")
print("   Drug space unified with MIMIC via fuzzy matching")
print("="*70)

STEP 1: Load MIMIC Dataset and Create Drug Vocabulary
MIMIC unique drugs: 1922
Canonical MIMIC drugs: 1790

STEP 2: Load eICU Data
diagnosis.csv loaded: (2710672, 7)
medication.csv loaded: (7301853, 15)
admissionDrug.csv loaded: (874920, 14)
Anemia patients in eICU: 5198

Extracting medications for anemia patients...
Total medication records: 364,734
After removing duplicates: 102,292
After filtering empty names: 102,292

STEP 3: Match eICU Drugs to MIMIC Vocabulary
Unique eICU drugs before matching: 2397


Matching eICU drugs to MIMIC: 100%|███████████████████████████████████████████████| 2397/2397 [00:08<00:00, 278.51it/s]


Matched 1806/2397 (75.3%)
After filtering to matched drugs: 88,003 interactions
After filtering drugs with <5 occurrences: 491 unique drugs
After filtering patients with <2 drugs: 4353 patients

eICU ANEMIA DATASET STATISTICS (MIMIC-Matched)
Total anemia patients: 4,353
Total interactions: 78,641
Unique drugs (MIMIC-matched): 491
Average drugs per patient: 18.07

First 10 rows:
     user                       item  rating
0  145270              acetaminophen       1
1  145270          calcium gluconate       1
2  145270    chlorhexidine gluconate       1
3  145270            famotidine (po)       1
4  145270           fentanyl citrate       1
5  145270                 furosemide       1
6  145270                 gabapentin       1
7  145270            hydralazine hcl       1
8  145270  hydrocodone-acetaminophen       1
9  145270          hydromorphone hcl       1

✓ Saved: C:\Users\Administrator\Desktop\medllm evn\Data\user_drug_rating_visit_eicu_anemia.csv

DRUG OVERLAP VERIFICATION
M